### 1.CV-Bench
它来自 Cambrian-1 研究，核心就是测试 VLM 的基础视觉能力。包含四种典型任务：
- 2D 空间关系：左、右、上、下等
- 物体数量识别
- 3D 深度顺序：哪个物体更靠近相机
- 相对距离：比较物体与相机或物体之间的距离关系
例如，给模型一张室内图片，提问：
图中的椅子和桌子，哪个距离摄像机更近？

优点： 有图像、问题、选项、正确答案，可直接计算 Accuracy。还可以分别加载 2D 和 3D 子集，不必进行额外标注。
局限： 主要是从 RGB 图像推断深度关系，而不是直接读取深度图；不考察多帧记忆。

In [8]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [9]:
import os
import huggingface_hub
from huggingface_hub import constants

print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))
print("实际使用:", constants.ENDPOINT)
print("Hub版本:", huggingface_hub.__version__)

# 检查是否配置代理
for key in ["HTTP_PROXY", "HTTPS_PROXY",
            "http_proxy", "https_proxy",
            "ALL_PROXY", "all_proxy"]:
    print(key, "已配置" if os.environ.get(key) else "未配置")

HF_ENDPOINT: https://hf-mirror.com
实际使用: https://hf-mirror.com
Hub版本: 0.36.2
HTTP_PROXY 未配置
HTTPS_PROXY 未配置
http_proxy 未配置
https_proxy 未配置
ALL_PROXY 未配置
all_proxy 未配置


In [10]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="nyu-visionx/CV-Bench",
    repo_type="dataset",
    local_dir="/home/qiuyuhang_26/workspace/VLM-test/dataset/CV-Bench"
)

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 1956.17it/s]


'/home/qiuyuhang_26/workspace/VLM-test/dataset/CV-Bench'

In [11]:
import io
import pandas as pd
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw

root = Path("/home/qiuyuhang_26/workspace/VLM-test/dataset/CV-Bench")
output = root / "samples"

for split in ["2d", "3d"]:
    # 读取 Parquet
    df = pd.read_parquet(root / f"test_{split}.parquet")

    # 去除重复图片，随机挑选30张
    df = df.drop_duplicates(subset="filename")
    samples = df.sample(n=30, random_state=42)

    save_dir = output / split.upper()
    save_dir.mkdir(parents=True, exist_ok=True)

    # 创建5列×6行的预览图
    sheet = Image.new("RGB", (1300, 1320), "white")
    draw = ImageDraw.Draw(sheet)

    records = []

    for i, (_, row) in enumerate(samples.iterrows()):
        # 从二进制数据还原图片
        img_bytes = row["image"]["bytes"]
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        # 保存原始分辨率图片
        filename = f"{split}_{i+1:02d}.png"
        img.save(save_dir / filename)

        # 生成缩略图
        thumb = ImageOps.contain(img, (240, 185))

        x = (i % 5) * 260
        y = (i // 5) * 220

        sheet.paste(thumb, (x + (260-thumb.width)//2, y))
        draw.text(
            (x + 10, y + 190),
            f"{split.upper()} #{i+1:02d}  {row['task']}",
            fill="black"
        )

        records.append({
            "image_file": filename,
            "task": row["task"],
            "question": row["question"],
            "answer": row["answer"]
        })

    # 保存总览图和对应题目
    sheet.save(output / f"{split}_overview.jpg")
    pd.DataFrame(records).to_csv(
        save_dir / "annotations.csv",
        index=False
    )

    print(f"{split.upper()}：已提取30张图片")

print("全部完成！")

2D：已提取30张图片
3D：已提取30张图片
全部完成！


### 2.SPAR-Bench：最契合你的空间与深度测试需求
数据集：SPAR-Bench · Tiny 版本 · Tiny RGB-D 版本

这是一个非常值得关注的数据集。相比 CV-Bench，它更强调真正的三维空间理解，包含：
- 深度预测（Depth）
- 距离预测（Distance）
- 摄像机运动推理（Camera Motion）
- 多视角位置匹配（Position Matching）
- 视角变化推理（View Change）
- 三维空间想象（Spatial Imagination）
其中尤其适合你的地方是：SPAR-Bench 有 RGB 和 RGB-D 两类数据版本。

RGB-D 版本还提供深度数值、相机位姿及相机内参，能够支持更严格的三维感知实验。Tiny 版本只包含 1,000 道题，每种任务 50 道，适合先快速比较模型。

注意：普通 VLM 往往只能接收深度图的可视化图像，而不能直接理解深度矩阵或相机内参。使用 RGB-D 版需要设计一致的输入方式。

In [12]:
snapshot_download(
    repo_id="jasonzhango/SPAR-Bench",
    repo_type="dataset",
    local_dir=str("/home/qiuyuhang_26/workspace/VLM-test/dataset/SPAR-Bench"),
    max_workers=4
)

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1143.43it/s]


'/home/qiuyuhang_26/workspace/VLM-test/dataset/SPAR-Bench'

### 3. VSI-Bench：最适合连续帧、第一视角与空间记忆
VLN 高相关

它与前两个数据集最大的区别是：输入来自真实室内环境的第一视角连续视频。

模型需要一边理解镜头移动，一边建立室内的空间结构。

例如，摄像头从走廊走进房间，先看到沙发，再转身看见桌子。即使沙发和桌子没有同时出现在一个画面中，模型也要理解它们之间的空间关系。

它能测出：
- 多帧信息能否整合
- 物体在不同视角下能否关联
- 是否记得先前看到的空间结构
- 能否估计房间尺寸、物体距离和空间布局
官方提供完整视频、问题标注和评测代码，并有一个用于减少非视觉捷径影响的 VSI-Bench-Debiased 子集。

对于 VLN，这比普通的视频事件问答（例如“刚才谁开了门”）更有针对性。

In [13]:
snapshot_download(
    repo_id="nyu-visionx/VSI-Bench",
    repo_type="dataset",
    local_dir="/home/qiuyuhang_26/workspace/VLM-test/dataset/VSI-Bench",
    max_workers=4
)

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 2392.23it/s]


'/home/qiuyuhang_26/workspace/VLM-test/dataset/VSI-Bench'